In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [ ]:
import os
from pathlib import Path

DATA_FILENAME = "QUAC 605 Project Data_processed.xlsx - 3 month_processed.csv"
DATA_PATH_CANDIDATES = (
    Path("Electric grid project") / "cv_QRC" / DATA_FILENAME,
    Path("projects/Electicty load prediction forecasting ")
    / "Electric grid project"
    / "cv_QRC"
    / DATA_FILENAME,
)
DATA_PATH = next((path for path in DATA_PATH_CANDIDATES if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        f"Place {DATA_FILENAME!r} in the repository's Electric grid project/cv_QRC directory. "
        f"Checked: {', '.join(str(path.resolve()) for path in DATA_PATH_CANDIDATES)}"
    )

df = pd.read_csv(DATA_PATH)
if os.getenv("NOTEBOOK_CI") == "1":
    df = df.head(600)
    print("NOTEBOOK_CI: using the first 600 rows")

print(f"Loaded {len(df)} rows from {DATA_PATH}")
df.head()


In [39]:

# feature column definition (9 total)

feature_cols = [
    "EMBEDDED_WIND_GENERATION",
    "EMBEDDED_SOLAR_GENERATION",
    "PUMP_STORAGE_PUMPING",
    "SCOTTISH_TRANSFER",
    "NEMO_FLOW",
    "ELECLINK_FLOW",
    "Temperature",
    "period_sin",
    "period_cos",
]

target_col = "ENGLAND_WALES_DEMAND"

# Extract input matrix X_raw and target y_raw

X_raw = df[feature_cols].values.astype(float)
y_raw = df[target_col].values.astype(float)

print("X_raw shape:", X_raw.shape)
print("y_raw shape:", y_raw.shape)

# Normalize X and y for CV encoding

X_scaler = StandardScaler()
X = X_scaler.fit_transform(X_raw)

y_scaler = StandardScaler()
y = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

print("X normalized shape:", X.shape)
print("y normalized shape:", y.shape)


X_raw shape: (4318, 9)
y_raw shape: (4318,)
X normalized shape: (4318, 9)
y normalized shape: (4318,)


In [40]:
# Define mode counts
n_modes_list = [4, 5, 6]

# Number of input features (9 total)
n_features = X.shape[1]
print("n_features:", n_features)

# mode count
n_modes_list = [6]

# Random generator (weights, reservoir phases, beam splitter angles, projections...etc)  #reproducibility
rng = np.random.default_rng(seed=42)

# projection matrix Dictionary (store)
W_enc = {}

# Projection matrices for each mode count
for k in n_modes_list:
    W_enc[k] = rng.normal(loc=0.0, scale=0.5, size=(k, n_features))
    print(f"W_enc[{k}] shape:", W_enc[k].shape)


n_features: 9
W_enc[6] shape: (6, 9)


In [41]:
# Apply random projection: X -> Z_k
def encode_with_projection(X, W):
    return (W @ X.T).T

Z = {}

for k in n_modes_list:
    Z[k] = encode_with_projection(X, W_enc[k])
    print(f"Z[{k}] shape:", Z[k].shape)


Z[6] shape: (4318, 6)


QRC

In [42]:


N_MODES = 6   # Set mode count per run

import pennylane as qml

# PennyLane's built-in Gaussian simulator
dev = qml.device("default.gaussian", wires=N_MODES, shots=1)

print("Ready with", N_MODES, "modes")



Ready with 6 modes


/usr/local/lib/python3.12/dist-packages/pennylane/devices/legacy_facade.py:177: PennyLaneDeprecationWarning: Setting shots on device is deprecated. Please use the `set_shots` transform on the respective QNode instead.
  warnings.warn(


In [43]:
# Hyperparameter: reservoir depth-number of Gaussian layers
DEPTH = 3  # 2 or 3

# Random reservoir parameters (fixed)
rng_res = np.random.default_rng(seed=123)

# Phase shift angles: shape (DEPTH, N_MODES)
phi = rng_res.normal(loc=0.0, scale=np.pi, size=(DEPTH, N_MODES))

# Beam splitter parameters: shape (DEPTH, N_MODES-1)
theta_bs = rng_res.normal(loc=0.0, scale=np.pi/4, size=(DEPTH, max(N_MODES-1, 1)))
phi_bs   = rng_res.normal(loc=0.0, scale=2*np.pi,   size=(DEPTH, max(N_MODES-1, 1)))

@qml.qnode(dev)
def qrc_circuit_single(z_t, meas_mode):
    """
    One QRC forward pass, for a single time step t and a single measured mode.
    z_t: array of shape (N_MODES,)
    meas_mode: integer index of the mode to measure (0 .. N_MODES-1)
    Returns: scalar <X> quadrature on that mode.
    """

    # Encoding: Displacement on each mode
    for i in range(N_MODES):
        qml.Displacement(z_t[i], 0.0, wires=i)

    # Reservoir: DEPTH Gaussian mixing layers
    for d in range(DEPTH):
        # Phase shifts
        for i in range(N_MODES):
            qml.Rotation(phi[d, i], wires=i)
        # Beamsplitters
        if N_MODES > 1:
            for i in range(N_MODES - 1):
                qml.Beamsplitter(theta_bs[d, i], phi_bs[d, i], wires=[i, i+1])

    # Readout: measure X quadrature (Quadrature operator)
    return qml.expval(qml.QuadX(meas_mode))


In [44]:
# features , projection matrix, z_shape

n_features = X.shape[1]
print("n_features:", n_features)

rng = np.random.default_rng(seed=42)

# Projection matrix for THIS mode count only
W_enc = rng.normal(loc=0.0, scale=0.5, size=(N_MODES, n_features))
print("W_enc shape:", W_enc.shape)

def encode_with_projection(X, W):
    return (W @ X.T).T  # gives shape (T, N_MODES)

# Z  (single array)
Z = encode_with_projection(X, W_enc)
print("Z shape:", Z.shape)


n_features: 9
W_enc shape: (6, 9)
Z shape: (4318, 6)


In [45]:
# Hidden State

def compute_hidden_states(Z):
    """
    Z: array of shape (T, N_MODES)
    Returns: H of shape (T, N_MODES) with hidden variables for each time step.
    """
    T = Z.shape[0]
    H = np.zeros((T, N_MODES))
    for t in range(T):
        z_t = Z[t]
        # compute hidden value for each mode
        for m in range(N_MODES):
            H[t, m] = qrc_circuit_single(z_t, m)
        if (t + 1) % 10 == 0:
            print(f"Computed hidden state {t+1}/{T}")
    return H

H = compute_hidden_states(Z)
print("H shape:", H.shape)


Computed hidden state 10/4318
Computed hidden state 20/4318
Computed hidden state 30/4318
Computed hidden state 40/4318
Computed hidden state 50/4318
Computed hidden state 60/4318
Computed hidden state 70/4318
Computed hidden state 80/4318
Computed hidden state 90/4318
Computed hidden state 100/4318
Computed hidden state 110/4318
Computed hidden state 120/4318
Computed hidden state 130/4318
Computed hidden state 140/4318
Computed hidden state 150/4318
Computed hidden state 160/4318
Computed hidden state 170/4318
Computed hidden state 180/4318
Computed hidden state 190/4318
Computed hidden state 200/4318
Computed hidden state 210/4318
Computed hidden state 220/4318
Computed hidden state 230/4318
Computed hidden state 240/4318
Computed hidden state 250/4318
Computed hidden state 260/4318
Computed hidden state 270/4318
Computed hidden state 280/4318
Computed hidden state 290/4318
Computed hidden state 300/4318
Computed hidden state 310/4318
Computed hidden state 320/4318
Computed hidden s

H_eff = H[0 : T-1]   #readout


y_eff = y[1 : T]     #target

Sanity Check!

In [46]:
#Check 1
print("X:", X.shape)
print("Z:", Z.shape)
print("H:", H.shape)
print("y:", y.shape)

print("\nAny NaN values?")
print("X:", np.isnan(X).any())
print("Z:", np.isnan(Z).any())
print("H:", np.isnan(H).any())
print("y:", np.isnan(y).any())

print("\nAny infinite values?")
print("X:", np.isinf(X).any())
print("Z:", np.isinf(Z).any())
print("H:", np.isinf(H).any())
print("y:", np.isinf(y).any())


X: (4318, 9)
Z: (4318, 6)
H: (4318, 6)
y: (4318,)

Any NaN values?
X: False
Z: False
H: False
y: False

Any infinite values?
X: False
Z: False
H: False
y: False


In [47]:
#Check 2
print("H mean:", H.mean())
print("H std:", H.std())
print("Per-mode std:", H.std(axis=0))


H mean: -0.004339732796466916
H std: 1.5537022211610103
Per-mode std: [1.14673118 1.48564175 1.81531877 1.69674181 1.33718444 1.73180228]
